In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("FeatureEngineering") \
    .master("local[*]") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print("Spark session created")

Spark session created


In [ ]:
import os
print(os.listdir("/content"))

['.config', 'sample_data']


In [ ]:
from google.colab import files
files.upload()

Saving us_accidents_parquet.zip to us_accidents_parquet.zip


In [ ]:
import zipfile
zip_path = "/content/us_accidents_parquet.zip"
extract_path = "/content/data"
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)
print("Unzipped successfully")

Unzipped successfully


In [ ]:
train_ready = spark.read.parquet("/content/data/train/")
val_ready   = spark.read.parquet("/content/data/val/")
test_ready  = spark.read.parquet("/content/data/test/")

In [ ]:
print("\nLOADING DATA\n")
train_ready = spark.read.parquet("/content/data/train/")
val_ready   = spark.read.parquet("/content/data/val/")
test_ready  = spark.read.parquet("/content/data/test/")
print("Train data loaded")
print("Validation data loaded")
print("Test data loaded")
print("\nROW COUNTS BEFORE DROP\n")
print("Train:", train_ready.count())
print("Validation:", val_ready.count())
print("Test:", test_ready.count())
print("\nDROPPING UNWANTED COLUMNS\n")
cols_to_drop = ["ID", "Description"]
train_ready = train_ready.drop(*cols_to_drop)
val_ready   = val_ready.drop(*cols_to_drop)
test_ready  = test_ready.drop(*cols_to_drop)
print("Columns dropped successfully")
print("\nROW COUNTS AFTER DROP\n")
print("Train:", train_ready.count())
print("Validation:", val_ready.count())
print("Test:", test_ready.count())
print("\nCURRENT COLUMNS\n")
print(train_ready.columns)
print("\nDATA PREPARATION COMPLETED\n")


LOADING DATA

Train data loaded
Validation data loaded
Test data loaded

ROW COUNTS BEFORE DROP

Train: 1447502
Validation: 361295
Test: 451689

DROPPING UNWANTED COLUMNS

Columns dropped successfully

ROW COUNTS AFTER DROP

Train: 1447502
Validation: 361295
Test: 451689

CURRENT COLUMNS

['Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)', 'Street', 'Side', 'City', 'County', 'State', 'Zipcode', 'Country', 'Timezone', 'Airport_Code', 'Weather_Timestamp', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop', 'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight']

DATA PREPARATION COMPLETED



In [ ]:
from pyspark.sql.functions import hour, dayofweek, month

train_ready = train_ready.withColumn("hour", hour("Start_Time")) \
                         .withColumn("day_of_week", dayofweek("Start_Time")) \
                         .withColumn("month", month("Start_Time"))

val_ready = val_ready.withColumn("hour", hour("Start_Time")) \
                     .withColumn("day_of_week", dayofweek("Start_Time")) \
                     .withColumn("month", month("Start_Time"))

test_ready = test_ready.withColumn("hour", hour("Start_Time")) \
                       .withColumn("day_of_week", dayofweek("Start_Time")) \
                       .withColumn("month", month("Start_Time"))

print("\nTime features added")

train_ready.select("Start_Time", "hour", "day_of_week", "month").show(5)


Time features added
+-------------------+----+-----------+-----+
|         Start_Time|hour|day_of_week|month|
+-------------------+----+-----------+-----+
|2016-05-26 16:13:46|  16|          5|    5|
|2016-05-26 17:00:25|  17|          5|    5|
|2016-05-26 20:05:53|  20|          5|    5|
|2016-05-26 20:05:53|  20|          5|    5|
|2016-05-27 05:07:17|   5|          6|    5|
+-------------------+----+-----------+-----+
only showing top 5 rows


In [ ]:
from pyspark.sql.functions import col
train_ready = train_ready.withColumn(
    "duration_minutes",
    (col("End_Time").cast("long") - col("Start_Time").cast("long")) / 60
)
val_ready = val_ready.withColumn(
    "duration_minutes",
    (col("End_Time").cast("long") - col("Start_Time").cast("long")) / 60
)
test_ready = test_ready.withColumn(
    "duration_minutes",
    (col("End_Time").cast("long") - col("Start_Time").cast("long")) / 60
)
print("\nDuration feature created")
train_ready.select("duration_minutes").show(5)


Duration feature created
+----------------+
|duration_minutes|
+----------------+
|           360.0|
|           360.0|
|           360.0|
|           360.0|
|           360.0|
+----------------+
only showing top 5 rows


In [ ]:
train_ready = train_ready.filter("duration_minutes > 0")
val_ready   = val_ready.filter("duration_minutes > 0")
test_ready  = test_ready.filter("duration_minutes > 0")
print("\nRemoved invalid durations")
print("Train rows:", train_ready.count())
print("Validation rows:", val_ready.count())
print("Test rows:", test_ready.count())


Removed invalid durations
Train rows: 1447502
Validation rows: 361295
Test rows: 451689


In [ ]:
from pyspark.ml.feature import StringIndexer
categorical_cols = ["State", "Weather_Condition"]
indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_index", handleInvalid="keep")
    for c in categorical_cols
]
print("\nCategorical indexers created")
print("Categorical columns:", categorical_cols)


Categorical indexers created
Categorical columns: ['State', 'Weather_Condition']


In [ ]:
label_indexer = StringIndexer(
    inputCol="Severity",
    outputCol="label"
)
print("\nLabel column created")


Label column created


In [ ]:
from pyspark.ml.feature import VectorAssembler
feature_cols = [
    "Distance(mi)", "Temperature(F)", "Humidity(%)",
    "Pressure(in)", "Visibility(mi)", "Wind_Speed(mph)",
    "hour", "day_of_week", "month", "duration_minutes",
    "State_index", "Weather_Condition_index"
]
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)
print("\Feature vector created")
print("Total features:", len(feature_cols))

\Feature vector created
Total features: 12


<>:12: SyntaxWarning: invalid escape sequence '\F'
<>:12: SyntaxWarning: invalid escape sequence '\F'
/tmp/ipython-input-425/1822935813.py:12: SyntaxWarning: invalid escape sequence '\F'
  print("\Feature vector created")


In [ ]:
from pyspark.ml import Pipeline
pipeline = Pipeline(stages=indexers + [label_indexer, assembler])
print("\Pipeline built")

\Pipeline built


<>:3: SyntaxWarning: invalid escape sequence '\P'
<>:3: SyntaxWarning: invalid escape sequence '\P'
/tmp/ipython-input-425/2164478271.py:3: SyntaxWarning: invalid escape sequence '\P'
  print("\Pipeline built")


In [ ]:
from pyspark.ml import Pipeline
pipeline = Pipeline(stages=indexers + [label_indexer, assembler])
print("Pipeline built")
pipeline_model = pipeline.fit(train_ready)
train_ready = pipeline_model.transform(train_ready)
val_ready   = pipeline_model.transform(val_ready)
test_ready  = pipeline_model.transform(test_ready)
print("\nData transformed")
train_ready.select("features", "label").show(5)

Pipeline built

Data transformed
+--------------------+-----+
|            features|label|
+--------------------+-----+
|[0.46299999999999...|  0.0|
|[0.002,73.0,44.0,...|  0.0|
|[0.0,65.0,58.0,29...|  0.0|
|[0.0,65.0,58.0,29...|  0.0|
|[0.02899999999999...|  0.0|
+--------------------+-----+
only showing top 5 rows


In [ ]:
from pyspark.ml.classification import RandomForestClassifier, GBTClassifier, NaiveBayes, LinearSVC
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

In [ ]:
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=50)
gbt = GBTClassifier(labelCol="label", featuresCol="features", maxIter=20)
nb = NaiveBayes(labelCol="label", featuresCol="features")
svm = LinearSVC(labelCol="label", featuresCol="features")

In [ ]:
models = {
    "Random Forest": rf,
    "Gradient Boosted Trees": gbt,
    "Naive Bayes": nb,
    "Linear SVM": svm
}

In [ ]:
evaluator_acc = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

evaluator_f1 = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

In [ ]:
from pyspark.sql.functions import udf
from pyspark.ml.linalg import Vectors, VectorUDT

def to_positive(v):
    return Vectors.dense([abs(x) for x in v])

abs_udf = udf(to_positive, VectorUDT())

train_ready = train_ready.withColumn("features_nb", abs_udf("features"))
test_ready  = test_ready.withColumn("features_nb", abs_udf("features"))

In [ ]:
nb = NaiveBayes(labelCol="label", featuresCol="features_nb")

In [34]:
rf_model = rf.fit(train_ready)
rf_preds = rf_model.transform(test_ready)
rf_acc = evaluator_acc.evaluate(rf_preds)
rf_f1 = evaluator_f1.evaluate(rf_preds)
print("Random Forest Accuracy:", round(rf_acc, 4))
print("Random Forest F1:", round(rf_f1, 4))
results.append(("Random Forest", rf_acc, rf_f1))
nb_model = nb.fit(train_ready)
nb_preds = nb_model.transform(test_ready)
nb_acc = evaluator_acc.evaluate(nb_preds)
nb_f1 = evaluator_f1.evaluate(nb_preds)
print("Naive Bayes Accuracy:", round(nb_acc, 4))
print("Naive Bayes F1:", round(nb_f1, 4))
results.append(("Naive Bayes", nb_acc, nb_f1))
svm_model = svm.fit(train_ready)
svm_preds = svm_model.transform(test_ready)
svm_auc = evaluator_bin.evaluate(svm_preds)
print("Linear SVM AUC:", round(svm_auc, 4))
results.append(("Linear SVM", svm_auc, 0))
gbt_model = gbt.fit(train_ready)
gbt_preds = gbt_model.transform(test_ready)
gbt_auc = evaluator_bin.evaluate(gbt_preds)
print("GBT AUC:", round(gbt_auc, 4))
results.append(("GBT", gbt_auc, 0))
print("\nFinal Results\n")
for r in results:
    print(r)
best_model = max(results, key=lambda x: x[1])
print("\nBest Model:", best_model[0], "Score:", round(best_model[1], 4))
print("\nCONFUSION MATRIX\n")
rf_preds.groupBy("label", "prediction").count().show()
print("\nSAMPLE PREDICTIONS\n")
rf_preds.select("label", "prediction").show(10)
print("\nFEATURE IMPORTANCE\n")
importances = rf_model.featureImportances
for i, imp in sorted(enumerate(importances), key=lambda x: x[1], reverse=True):
    print(feature_cols[i], ":", round(imp, 4))
rf_preds.select("label", "prediction") \
    .toPandas() \
    .to_csv("/content/final_predictions.csv", index=False)
print("Predictions saved successfully")

Random Forest Accuracy: 0.931
Random Forest F1: 0.9012
Naive Bayes Accuracy: 0.1962
Naive Bayes F1: 0.3046
Linear SVM AUC: 0.6387
GBT AUC: 0.8901

Final Results

('Random Forest', 0.9310056255520944, 0.9012356977927326)
('Random Forest', 0.9310056255520944, 0.9012356977927326)
('Naive Bayes', 0.19620136864081258, 0.3046164353882282)
('Linear SVM', 0.6386620079500313, 0)
('GBT', 0.8900620832924496, 0)

Best Model: Random Forest Score: 0.931

CONFUSION MATRIX

+-----+----------+------+
|label|prediction| count|
+-----+----------+------+
|  2.0|       0.0| 13244|
|  1.0|       1.0|  1641|
|  0.0|       1.0|   276|
|  1.0|       0.0| 12799|
|  3.0|       1.0|    58|
|  2.0|       1.0|   128|
|  0.0|       0.0|418884|
|  3.0|       0.0|  4659|
+-----+----------+------+


SAMPLE PREDICTIONS

+-----+----------+
|label|prediction|
+-----+----------+
|  0.0|       0.0|
|  1.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  2.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  1.0|      

In [35]:
print("\nCONFUSION MATRIX\n")
rf_preds.groupBy("label", "prediction").count().show()


CONFUSION MATRIX

+-----+----------+------+
|label|prediction| count|
+-----+----------+------+
|  2.0|       0.0| 13244|
|  1.0|       1.0|  1641|
|  0.0|       1.0|   276|
|  1.0|       0.0| 12799|
|  3.0|       1.0|    58|
|  2.0|       1.0|   128|
|  0.0|       0.0|418884|
|  3.0|       0.0|  4659|
+-----+----------+------+



In [36]:
print("\nSAMPLE PREDICTIONS\n")
rf_preds.select("label", "prediction").show(10)


SAMPLE PREDICTIONS

+-----+----------+
|label|prediction|
+-----+----------+
|  0.0|       0.0|
|  1.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  2.0|       0.0|
|  0.0|       0.0|
|  0.0|       0.0|
|  1.0|       0.0|
|  0.0|       0.0|
|  1.0|       0.0|
+-----+----------+
only showing top 10 rows


In [37]:
print("\nFEATURE IMPORTANCE\n")
importances = rf_model.featureImportances
for i, imp in sorted(enumerate(importances), key=lambda x: x[1], reverse=True):

    print(feature_cols[i], ":", round(imp, 4))


FEATURE IMPORTANCE

duration_minutes : 0.458
State_index : 0.2945
Distance(mi) : 0.0925
month : 0.0911
Pressure(in) : 0.0324
Weather_Condition_index : 0.0104
Humidity(%) : 0.0084
hour : 0.0071
Temperature(F) : 0.003
Wind_Speed(mph) : 0.0018
Visibility(mi) : 0.0005
day_of_week : 0.0002


In [38]:
print("\nSAVING PREDICTIONS\n")
(
    rf_preds.select("label", "prediction")
    .toPandas()    .to_csv("/content/final_predictions.csv", index=False)
)
print("Predictions saved successfully")


SAVING PREDICTIONS

Predictions saved successfully


In [39]:
print("\nSAVING FINAL DATASETS (PARQUET)\n")
train_ready.write.mode("overwrite").parquet("/content/final_data/train/")
val_ready.write.mode("overwrite").parquet("/content/final_data/val/")
test_ready.write.mode("overwrite").parquet("/content/final_data/test/")
print("Datasets saved successfully")


SAVING FINAL DATASETS (PARQUET)

Datasets saved successfully


In [40]:
import os
print("\nVERIFY SAVED DATA\n")
print(os.listdir("/content/final_data"))


VERIFY SAVED DATA

['val', 'test', 'train']


In [41]:
print("\nPIPELINE COMPLETED SUCCESSFULLY\n")


PIPELINE COMPLETED SUCCESSFULLY



In [42]:
"/content/final_predictions.csv"

'/content/final_predictions.csv'

In [43]:
"/content/final_data/train/"
"/content/final_data/val/"
"/content/final_data/test/"

'/content/final_data/test/'

In [44]:
import shutil
shutil.make_archive('/content/final_data', 'zip', '/content/final_data')
from google.colab import files
files.download('/content/final_data.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [47]:
state_accidents = train_ready.groupBy("State").count()
state_accidents.toPandas().to_csv("/content/accidents_by_state.csv", index=False)
hour_accidents = train_ready.groupBy("hour").count()
hour_accidents.toPandas().to_csv("/content/accidents_by_hour.csv", index=False)
weather_accidents = train_ready.groupBy("Weather_Condition").count()
weather_accidents.toPandas().to_csv("/content/accidents_by_weather.csv", index=False)
print("Saved business insight CSVs")

Saved business insight CSVs
